In [1]:
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(
    weights='imagenet', # start from filters learned on ImageNet
    include_top=False, # drop the original 1000-class head
    input_shape=(96, 96, 3) # image size going in
)
base.trainable = False  # freeze: training can't change these weights

print(base.output_shape)
print(base.count_params())

c:\Users\Edward\Desktop\python\Lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


(None, 3, 3, 1280)
2257984


In [2]:
from tensorflow.keras import layers, Sequential

tl_model = Sequential([
    layers.Input(shape=(32, 32, 3)),            # one CIFAR image
    layers.Resizing(96, 96),                    # 32x32 -> 96x96
    layers.Rescaling(scale=1/127.5, offset=-1), # 0..255 -> -1..1
    base,                                       # frozen MobileNetV2 from before
    layers.GlobalAveragePooling2D(),            # 3x3x1280 -> 1280 numbers
    layers.Dense(10, activation='softmax')      # 10 CIFAR classes
])

tl_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.callbacks import EarlyStopping

#data: raw pixels 0-255 (Rescaling inside the model does the scaling)
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
y_train, y_test = y_train.ravel(), y_test.ravel()   # (N,1) -> (N,)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

tl_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#sanity check on 1000 training images: head is untrained
print(tl_model.evaluate(X_train[:1000], y_train[:1000]))

#train only the head
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

tl_history = tl_model.fit(
    X_train[:10000], y_train[:10000],
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

print(tl_model.evaluate(X_test, y_test))

(50000, 32, 32, 3) (50000,) (10000, 32, 32, 3) (10000,)
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 52ms/step - accuracy: 0.8810 - loss: 0.3336
[0.3336389362812042, 0.8809999823570251]
Epoch 1/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 40s 262ms/step - accuracy: 0.8804 - loss: 0.3474 - val_accuracy: 0.8810 - val_loss: 0.3545
Epoch 2/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 58s 416ms/step - accuracy: 0.9002 - loss: 0.2980 - val_accuracy: 0.8890 - val_loss: 0.3422
Epoch 3/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 15s 104ms/step - accuracy: 0.9139 - loss: 0.2591 - val_accuracy: 0.8810 - val_loss: 0.3541
Epoch 4/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 15s 104ms/step - accuracy: 0.9240 - loss: 0.2349 - val_accuracy: 0.8810 - val_loss: 0.3538
313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 50ms/step - accuracy: 0.8530 - loss: 0.4306
[0.4306279420852661, 0.8529999852180481]
